Import statments

In [12]:
import numpy as np
import pandas as pd
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn. ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [13]:
path = kagglehub.dataset_download("yashdevladdha/uber-ride-analytics-dashboard")

print(path)

C:\Users\finna\.cache\kagglehub\datasets\yashdevladdha\uber-ride-analytics-dashboard\versions\2


Print table before modifications

In [14]:
ls C:\Users\finna\.cache\kagglehub\datasets\yashdevladdha\uber-ride-analytics-dashboard\versions\2

 Volume in drive C is Windows
 Volume Serial Number is 46D8-AD3C

 Directory of C:\Users\finna\.cache\kagglehub\datasets\yashdevladdha\uber-ride-analytics-dashboard\versions\2

10/14/2026  03:35 PM    <DIR>          .
10/14/2026  03:35 PM    <DIR>          ..
10/14/2026  03:35 PM         4,066,285 Dasboard.gif
10/14/2026  03:35 PM        25,537,048 ncr_ride_bookings.csv
10/14/2026  03:35 PM         7,861,102 Uber.pbix
               3 File(s)     37,464,435 bytes
               2 Dir(s)  577,389,252,608 bytes free


In [15]:
table = pd.read_csv("C:/Users/finna/.cache/kagglehub/datasets/yashdevladdha/uber-ride-analytics-dashboard/versions/2/ncr_ride_bookings.csv")

table_copy = table.copy()

display(table_copy.head())

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


Getting rid of unncessary features/features that are dead give aways

In [16]:
unncessary_feats = ['Booking ID', 'Customer ID', 'Booking Status', 'Cancelled Rides by Customer', "Reason for cancelling by Customer", 'Cancelled Rides by Driver', 'Driver Cancellation Reason']
table_copy.drop(unncessary_feats, axis=1, inplace=True)

display(table_copy.head())

,Date,Time,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,737.0,48.21,4.1,4.3,UPI


Time

In [19]:
table_copy['Date'] = pd.to_datetime(table_copy['Date'])
table_copy['Time'] = pd.to_datetime(table_copy['Time'], format='%H:%M:%S')
table_copy['hour'] = table_copy['Time'].dt.hour
table_copy['day_of_week'] = table_copy['Date'].dt.dayofweek
table_copy.drop('Date', axis=1, inplace=True)
table_copy.drop('Time', axis=1, inplace=True)

display(table_copy.head())

,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,hour,day_of_week
0,eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,5
1,Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,18,4
2,Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card,8,4
3,Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,416.0,34.02,4.6,5.0,UPI,17,0
4,Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,737.0,48.21,4.1,4.3,UPI,22,0


There are only 6 types of vehicles, could one hot encode
Also need to fill in missing values for booking value, ride distance, driver ratings, customer rating, and ride distance